# Detekcja krawędzi

## Cel ćwiczenia

- Zapoznanie z metodami detekcji krawędzi:
    - Sobel, Prewitt, Roberts - przypomnienie,
    - Laplasjan z Gaussa (LoG – ang. Laplacian of Gaussian),
    - Canny.

Detekcja krawędzi przez wiele lat była podstawą algorytmów segmentacji.
Krawędzie wykrywane są najczęściej z wykorzystaniem pierwszej (gradient) i drugiej (Laplasjan) pochodnej przestrzennej.
Wykorzystanie obu metod zaprezentowane zostało w ćwiczeniu *Przetwarzanie wstępne. Filtracja kontekstowa*.

W niniejszym ćwiczeniu poznane detektory krawędzi zostaną porównane z bardziej zaawansowanymi: Laplasjan z funkcji Gaussa (LoG), Zero Crossing i Canny.

## Laplasjan z Gaussa (LoG)

Funkcja Gaussa:<br>
\begin{equation}
h(r) = e^{\frac{-r^2}{2 \sigma^2}}
\end{equation}<br>
gdzie:
- $r^2 = x^2 + y^2$
- $\sigma$ to odchylenie standardowe.

Działanie filtracji Gaussowskiej zostało przedstawione w ćwiczeniu "Przetwarzanie wstępne". W jej wyniku następuje rozmazanie obrazu.
Laplasjan tej funkcji dany jest wzorem:

\begin{equation}
\nabla^2 h(r) = \frac{r^2 - 2\sigma^2}{\sigma^4} e^{-\frac{r^2}{2\sigma^2}}
\end{equation}

Funkcję (z oczywistych powodów) nazywamy Laplasjan z Gaussa (LoG).
Ponieważ druga pochodna jest operacją liniową, konwolucja obrazu z $\nabla^2 h(r)$ daje taki sam efekt jak zastosowanie filtracji Gaussa na obrazie, a następnie obliczenie Laplasjanu z wyniku.
Lokalizacja krawędzi polega na znalezieniu miejsca, gdzie po filtracji LoG następuje zmiana znaku.

1. Wczytaj obraz *house.png*.
2. Wykonaj rozmycie Gaussowskie obrazu wejściowego.
W tym celu wykorzysaj funkcję `cv2.GaussianBlur(img, kSize, sigma)`.
Pierwszy argument jest obrazem wejśćiowym.
Drugi jest rozmiarem filtru (podanym w nawiasach okrągłych, np. *(3, 3)*).
Trzecim argumentem jest odchylenie standardowe. Wartość jest dobrana automatycznie, jeśli zosanie podana wartość `0` (będą równe rozmiarowi).
3. Oblicz laplasjan obrazu rozmytego.
W tym celu wykorzysaj funkcję `cv2.Laplacian(img, ddepth)`.
Pierszym argumentem jest obraz wejściowy.
Drugim argumentem jest typ danych wejściowych. Użyj `cv2.CV_32F`.
4. Wyznacz miejsca zmiany znaku.
Zaimplementuj funkcję `crossing(LoG, thr)`:
    - Najpierw stwórz tablicę, do której zostanie zapisany wynik.
    Jej rozmiar jest taki sam jak przetwarzanego obrazu.
    - Następnie wykonaj pętle po obrazie (bez ramki jednopikselowej).
    W każdej iteracji stwórz otoczenie o rozmiarze $3 \times 3$.
    Dla otoczenia oblicz wartość maksymalną i minimalną.
    - Jeśli wartości te mają przeciwne znaki, to do danego miejsca tablicy przypisz wartość:
        - jeśli piksel wejściowy > 0, to dodaj do niego wartość bezwzględną minimum.
        - jeśli piksel wejściowy < 0, to do jego wartości bezwzględnej dodaj maksimum.
    - Zmień zakres wykonanej tablicy do $<0, 255>$.
    - Wykonaj progowanie tablicy. Próg jest argumentem wejściowym.
    - Przeskaluj dane binarne do wartości `[0, 255]`.
    - Wykonaj konwersję do typu *uint8*.
    - Wykonaj rozmycie medianowe wyniku.
    Wykorzystaj funkcję `cv2.medianBlur(img, kSize)`.
    Pierwszym argumentem jest obraz wejśćiowy, a drugim rozmiar filtra.
    - Zwróć wyznaczoną tablicę.
5. Wyświetl obraz wynikowy.
6. Dobierz parametry (rozmiar filtru Gaussa, odchylenie standardowe, próg binaryzacji) tak, by widoczne były kontury domu, ale nie dachówki.

In [ ]:
import cv2
from matplotlib import pyplot as plt
import numpy as np
import math
import os
import requests

url = 'https://raw.githubusercontent.com/vision-agh/poc_sw/master/09_Canny/'

fileNames = ["dom.png"]
for fileName in fileNames:
  if not os.path.exists(fileName):
      r = requests.get(url + fileName, allow_redirects=True)
      open(fileName, 'wb').write(r.content)



In [ ]:
house =cv2.imread('dom.png', cv2.IMREAD_GRAYSCALE)
houseBlur=cv2.GaussianBlur(house, (5,5), 1) #rozmycie szumu
laplasjan=cv2.Laplacian(houseBlur, cv2.CV_32F) #+ obraz z jasnego do ciemnego, - obraz z ciemnego do jasnego, zmiana znaku to mozliwa krawedz
def crossing(LoG, thr):
  h,w =LoG.shape
  result = np.zeros((h,w), dtype=np.uint8)
  for y in range(1, h-1):
    for x in range(1, w-1):
      otoczenie=LoG[y-1:y+2, x-1:x+2] #otoczenie 3x3
      min_val=np.min(otoczenie)
      max_val=np.max(otoczenie)
      if min_val* max_val<0: # jesli w oknie +,- to nastapila zmiana znakow i mozliwa krawedz
        if(LoG[y,x]>0): # jesli srodek dodatni to uzywamy min
          result[y,x]=LoG[y,x]+abs(min_val)
        elif(LoG[y,x]<0): # jesli srodek ujemny to liczymy sume z najwyzszego piksela w okolicy
          result[y,x]=abs(LoG[y,x])+max_val
        #dzieki temu mamy wyznaczona mockrawedzi (wiemy czy jest to szum czy niei)
  result_norm=cv2.normalize(result,None, 0, 255, cv2.NORM_MINMAX)
  result_bin= (result_norm >thr).astype(np.uint8) *255
  result_blur=cv2.medianBlur(result_bin, 3)
  return result_blur

In [ ]:
obraz1=crossing(laplasjan, 40)
plt.figure(figsize=(10,5))
plt.subplot(1,2,1)
plt.imshow(house, cmap='gray')
plt.subplot(1,2,2)
plt.imshow(obraz1, cmap='gray')

## Algorytm Canny'ego

> Algorytm Canny'ego to często wykorzystywana metoda detekcji krawędzi.
> Zaproponowana została w 1986r. przez Johna F. Cannego.
> Przy jego projektowaniu założono trzy cele:
> - niska liczba błędów - algorytm powinien znajdywać wszystkie krawędzie oraz generować jak najmniej fałszywych detekcji,
> - punkty krawędziowe powinny być poprawnie lokalizowane - wykryte punkty powinny być jak najbardziej zbliżone do rzeczywistych,
> - krawędzie o szerokości 1 piksela - algorytm powinien zwrócić jeden punkt dla każdej rzeczywistej krawędzi.

Zaimplementuj algorytm detekcji krawędziCanny'ego:
1. W pierwszym kroku obraz przefiltruj dwuwymiarowym filtrem Gaussa.
2. Następnie oblicz gradient pionowy i poziomy ($g_x $ i $g_y$).
Jedną z metod jest filtracja Sobela.
3. Dalej oblicz amplitudę:
$M(x,y)  = \sqrt{g_x^2+g_y^2}$ oraz kąt:
$\alpha(x,y) = arctan(\frac{g_y}{g_x})$.
Do obliczenia kąta wykorzystaj funkcję `np.arctan2(x1, x2)`.
Wynik jest w radianach.
4. W kolejnym etapie wykonaj kwantyzację kątów gradientu.
Kąty od $-180^\circ$ do $180^\circ$ można podzielić na 8 przedziałów:
[$-22.5^\circ, 22.5^\circ$], [$22.5^\circ, 67.5^\circ$],
[$67.5^\circ, 112.5^\circ$], [$112.5^\circ, 157.5^\circ$],
[$157.5^\circ, -157.5^\circ$], [$-157.5^\circ, -112.5^\circ$],
[$-112.5^\circ, -67.5^\circ$], [$-67.5^\circ, -22.5^\circ$].
Przy czym należy rozpatrywać tylko 4 kierunki:
    - pionowy ($d_1$),
    - poziomy ($d_2$),
    - skośny lewy ($d_3$),
    - skośny prawy ($d_4$).
5. Dalej przeprowadź eliminację pikseli, które nie mają wartości maksymalnej (ang. *nonmaximal suppresion*).
Celem tej operacji jest redukcja szerokości krawędzi do rozmiaru 1 piksela.
Algorytm przebiega następująco.
W rozpatrywanym otoczeniu o rozmiarze $3 \times 3$:
    - określ do którego przedziału należy kierunek gradientu piksela centralnego ($d_1, d_2, d_3, d_4$).
    - przeanalizuje sąsiadów leżących na tym kierunku.
Jeśli choć jeden z nich ma amplitudę większą niż piksel centralny, to należy uznać, że nie jest lokalnym maksimum i do wyniku przypisać $g_N(x,y) = 0$.
W przeciwnym przypadku $g_N(x,y) = M(x,y)$.
Przez $g_N$ rozumiemy obraz detekcji lokalnych maksimów.
Zaimplementuj funkcję `nonmax`.
Pierwszym argementem jest macierz kierunków (po kwantyzacji).
Drugim argumentem jest macierz amplitudy.
6. Ostatnią operacją jest binaryzacja obrazu $g_N$.
Stosuje się tutaj tzw. binaryzację z histerezą.
Wykorzystuje się w niej dwa progi: $T_L$ i $T_H$, przy czym $T_L < T_H$.
Canny zaproponował, aby stosunek progu wyższego do niższego był jak 3 lub 2 do 1.
Rezultaty binaryzacji można opisać jako:<br>
$g_{NH}(x,y) = g_N(x,y) \geq TH $<br>
$g_{NL}(x,y) = TH > g_N(x,y) \geq TL $<br>
Można powiedzieć, że na obrazie $g_{NH}$ są "pewne" krawędzie.
Natomiast na $g_{NL}$ "potencjalne".
Często krawędzie "pewne" nie są ciągłe.
Dlatego wykorzystuje się obraz $g_{NL}$ w następującej procedurze:
    - Stwórz stos zawierający wszystkie piksele zaznaczone na obrazie $g_{NH}$.
W tym celu wykorzystaj listę współrzędnych `[row, col]`.
Do pobrania elementu z początku służy metoda `list.pop()`.
Do dodania elementu na koniec listy służy metoda `list.append(new)`.
    - Stwórz obraz, który będzie zawierał informację czy dany piksel został już odwiedzony.
    - Stwórz obraz, któy zawierać będzie wynikowe krawędzie.
Jej rozmiar jest równy rozmiarowi obrazu.
    - Wykonaj pętlę, która będzie pobierać elementy z listy, dopóki ta nie będzie pusta.
W tym celu najlepiej sprawdzi się pętla `while`.
    - W każdej iteracji pobierz element ze stosu.
    - Sprawdź czy dany element został już odwiedzony.
    - Jeśli nie został, to:
        - Oznacz go jako odwiedzony,
        - Oznacz piksel jako krawędź w wyniku,
        - Sprawdź otoczenie piksela w obrazie $g_{NL}$,
        - Dodaj do stosu współrzędne otoczenia, które zawierają krawędź (potencjalną).
        Można to wykoanać np. pętlą po stworzonym otoczeniu.
7. Wyświetl obraz oryginalny, obraz $g_{NH}$ oraz obraz wynikowy.

Pomocnicze obrazy $g_{NH}$ i $g_{NL}$ zostały wprowadzone dla uproszczenia opisu.
Algorytm można zaimplementować wbardziej "zwarty" sposób.

Na podstawie powyższego opisu zaimplementuj algorytm Cannego.

In [ ]:
def nomax(alpha, mag):
    h, w = mag.shape
    Z = np.zeros((h, w), dtype=np.float32)
    angle = alpha * 180 / np.pi  #radiany na stopnie
    angle[angle < 0] += 180      #przedzial 0-18
    for y in range(1, h - 1):
        for x in range(1, w - 1):
            q = 0.0  #piksel po jednej stronie gradientu
            r = 0.0  #piksel po drugiej stronie gradientu
            #poziom
            if (0 <= angle[y, x] < 22.5) or (157.5 <= angle[y, x] < 180):
                q = mag[y, x + 1]
                r = mag[y, x - 1]
            #skos/
            elif 22.5 <= angle[y, x] < 67.5:
                q = mag[y + 1, x - 1]
                r = mag[y - 1, x + 1]
            #pion
            elif 67.5 <= angle[y, x] < 112.5:
                q = mag[y + 1, x]
                r = mag[y - 1, x]
            #skos\
            elif 112.5 <= angle[y, x] < 157.5:
                q = mag[y - 1, x - 1]
                r = mag[y + 1, x + 1]
            #jesli jest maks wsrod sasiadow
            if (mag[y, x] >= q) and (mag[y, x] >= r):
                Z[y, x] = mag[y, x]
            else: #jesli nie ma maks
                Z[y, x] = 0.0
    return Z


def histereza(img, slaba, silna=255):
    rows, cols = img.shape
    table_0 = np.zeros_like(img, dtype=np.uint8)
    stos = []
    for x in range(1, rows - 1):
        for y in range(1, cols - 1):
            if img[x, y] == silna:
                table_0[x, y] = 255
                stos.append((x, y))
    while(stos):
        i, j = stos.pop()
        for x in range(i - 1, i + 2):
            for y in range(j - 1, j + 2):
                if 0 <= x < rows and 0 <= y < cols:
                    if img[x, y] == slaba and table_0[x, y] == 0:
                        table_0[x, y] = 255
                        stos.append((x, y))
    return table_0
def canny_algorithm(img, gauss_window=5, gauss_sigma=1, thr1_high=0.8, thr2_low=0.4):
    img_blur = cv2.GaussianBlur(img, (gauss_window, gauss_window), gauss_sigma)
    g_x = cv2.Sobel(img_blur, cv2.CV_32F, 1, 0, ksize=3)
    g_y = cv2.Sobel(img_blur, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(g_x**2 + g_y**2)
    alpha = np.arctan2(g_y, g_x)
    nms = nomax(alpha, mag)
    max_val = np.max(nms)
    th_H = thr1_high * max_val #wartosci powyzej th_H -silne krawedzie
    th_L = thr2_low * max_val #wartosci miedzy th_L a th_H slabe krawedzie
    thresh_ = np.zeros_like(nms, dtype=np.uint8)
    slaba_kraw = 50
    mocna_kraw = 255
    mocna_x, mocna_y = np.where(nms >= th_H)
    slaba_x, slaba_y = np.where((nms >= th_L) & (nms < th_H))
    thresh_[mocna_x, mocna_y] = mocna_kraw
    thresh_[slaba_x, slaba_y] = slaba_kraw
    result_img = histereza(thresh_, slaba_kraw, mocna_kraw)
    return result_img, nms


canny_result, nms = canny_algorithm(house, gauss_window=3, gauss_sigma=0, thr1_high=0.2, thr2_low=0.1)

plt.figure(figsize=(10,5))
plt.subplot(1,3,1)
plt.imshow(house, cmap='gray')
plt.title("oryginal")
plt.subplot(1,3,2)
plt.title("przed histereza")
plt.imshow(nms, cmap='gray')
plt.subplot(1,3,3)
plt.imshow(canny_result, cmap='gray')
plt.title("koncowy obraz")


## Algorytm Canny'ego - OpenCV

1. Wykonaj dektekcję krawędzi metodą Canny'ego wykorzystując funkcję `cv2.Canny`.
    - Pierwszym argumentem funkcji jest obraz wejściowy.
    - Drugim argumentem jest mniejszy próg.
    - Trzecim argumentem jest większy próg.
    - Czwarty argument to tablica, do której wpisany zostanie wynik.
    Można zwrócić go przez wartość i podać wartość `None`.
    - Piąty argument to rozmiar operatora Sobela (w naszym przypadku 3).
    - Szósty argument to rodzaj używanej normy.
    0 oznacza normę $L_1$, 1 oznacza normę $L_2$. Użyj $L_2$.
2. Wynik wyświetl i porównaj z własną implementacją.

In [ ]:
canny_ = cv2.Canny(house, 70, 190, apertureSize=3, L2gradient=True)
plt.figure(figsize=(10,10))
plt.subplot(1,2,1)
plt.title("canny z biblioteki")
plt.imshow(canny_, cmap='gray')

plt.subplot(1,2,2)
plt.title(" implementacja")
plt.imshow(canny_result, cmap='gray')